In [11]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

DATABASE_CONFIG = {
    'host': 'localhost',
    'database': 'breathwell_clinic',
    'user': 'postgres',
    'password': '0702',
    'port': '5432'
}

connection_string = (
    f"postgresql://{DATABASE_CONFIG['user']}:{DATABASE_CONFIG['password']}"
    f"@{DATABASE_CONFIG['host']}:{DATABASE_CONFIG['port']}"
    f"/{DATABASE_CONFIG['database']}"
)

engine = create_engine(connection_string)

df = pd.read_sql_table('asthma_patients_data', engine)

print("початкова інформація про датасет:")
print(f"розмір: {df.shape}")
print(f"кількість записів: {df.shape[0]}")
print(f"кількість ознак: {df.shape[1]}")
print(f"\nтипи даних:")
print(df.dtypes)
print(f"\nперші 5 рядків:")
display(df.head())

print("\n" + "="*80)
print("1a. аналіз та заміна пропущених значень")
print("="*80)

missing_data = pd.DataFrame({
    'стовпець': df.columns,
    'пропущені': df.isnull().sum(),
    'відсоток': (df.isnull().sum() / len(df) * 100).round(2)
})

print("\nінформація про пропущені значення:")
display(missing_data[missing_data['пропущені'] > 0])

total_missing = df.isnull().sum().sum()
if total_missing == 0:
    print("\nвисновок: пропущені значення відсутні")
    print("не потрібно застосовувати методи імпутації")
else:
    print(f"\nзагальна кількість пропущених значень: {total_missing}")

df_clean = df.copy()

print("\n" + "="*80)
print("1b. кодування категоріальних ознак")
print("="*80)

categorical_cols = df_clean.select_dtypes(include=['object', 'bool']).columns.tolist()

if 'timestamp' in categorical_cols:
    categorical_cols.remove('timestamp')

print(f"\nкатегоріальні стовпці ({len(categorical_cols)}):")

binary_cols = []
nominal_cols = []

for col in categorical_cols:
    n_unique = df_clean[col].nunique()
    unique_vals = df_clean[col].unique()
    
    print(f"\n{col}:")
    print(f"  кількість унікальних значень: {n_unique}")
    print(f"  значення: {unique_vals}")
    
    if n_unique == 2:
        binary_cols.append(col)
        print(f"  → буде застосовано label encoding")
    elif n_unique > 2:
        nominal_cols.append(col)
        print(f"  → буде застосовано one-hot encoding")

print(f"\n\nпідсумок:")
print(f"бінарні ознаки (label encoding): {binary_cols}")
print(f"номінальні ознаки (one-hot encoding): {nominal_cols}")

encoding_changes = []

le = LabelEncoder()

for col in binary_cols:
    before_sample = df_clean[col].head(10).tolist()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    after_sample = df_clean[col].head(10).tolist()
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    
    print(f"\n{col} → {mapping}")
    
    encoding_changes.append({
        'стовпець': col,
        'тип_кодування': 'label encoding',
        'приклад_до': str(before_sample[:3]),
        'приклад_після': str(after_sample[:3])
    })

if nominal_cols:
    print(f"\nстворення dummy змінних для: {nominal_cols}")
    before_cols = df_clean.shape[1]
    df_clean = pd.get_dummies(df_clean, columns=nominal_cols, drop_first=True)
    after_cols = df_clean.shape[1]
    print(f"параметр drop_first=True використано для уникнення мультиколінеарності")
    print(f"створено нових стовпців: {after_cols - before_cols + len(nominal_cols)}")

print(f"\nрозмір до кодування: {df.shape}")
print(f"розмір після кодування: {df_clean.shape}")
print(f"зміна кількості стовпців: {df_clean.shape[1] - df.shape[1]:+d}")

print("\nтаблиця змін при кодуванні:")
display(pd.DataFrame(encoding_changes))

print("\nпорівняння датасетів (перші 5 рядків):")
print("до кодування:")
display(df[binary_cols + nominal_cols].head())
print("після кодування:")
display(df_clean[[col for col in df_clean.columns if any(orig in col for orig in binary_cols + nominal_cols)]].head())

print("\n" + "="*80)
print("1c. опрацювання аномальних значень")
print("="*80)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

exclude_cols = ['timestamp', 'diagnosis', 'patientid']
binary_categorical_cols = ['gender', 'smoking', 'ethnicity', 'educationlevel', 
                           'petallergy', 'familyhistoryasthma', 'historyofallergies', 
                           'eczema', 'hayfever', 'gastroesophagealreflux', 
                           'wheezing', 'shortnessofbreath', 'chesttightness', 
                           'coughing', 'nighttimesymptoms', 'exerciseinduced']

numeric_cols = [col for col in numeric_cols 
                if col.lower() not in exclude_cols 
                and col not in binary_categorical_cols]

print(f"\nчислові ознаки для аналізу аномалій ({len(numeric_cols)}):")
print(numeric_cols)
print(f"\nвиключено з аналізу:")
print(f"- ідентифікатори: {[col for col in exclude_cols if col in df_clean.columns]}")
print(f"- бінарні/категоріальні: {binary_categorical_cols}")

print("\nметод: міжквартільний розмах (IQR)")
print("формула: [Q1 - 1.5*IQR, Q3 + 1.5*IQR]")

df_no_outliers = df_clean.copy()
outliers_info = []

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers_mask = (df_clean[col] < lower) | (df_clean[col] > upper)
    n_outliers = outliers_mask.sum()
    
    if n_outliers > 0:
        outliers_info.append({
            'ознака': col,
            'Q1': round(Q1, 2),
            'Q3': round(Q3, 2),
            'IQR': round(IQR, 2),
            'нижня_межа': round(lower, 2),
            'верхня_межа': round(upper, 2),
            'аномалій': n_outliers,
            'відсоток': round(n_outliers / len(df_clean) * 100, 2)
        })
        
        df_no_outliers[col] = np.clip(df_no_outliers[col], lower, upper)

if outliers_info:
    outliers_df = pd.DataFrame(outliers_info)
    print("\nвиявлені аномалії:")
    display(outliers_df)
    print(f"\nметод обробки: обрізання (capping)")
    print("аномальні значення замінено на граничні допустимі")
    
    print("\nпорівняння значень до та після обробки аномалій (перші 3 ознаки):")
    comparison_cols = [item['ознака'] for item in outliers_info[:3]]
    if comparison_cols:
        print("\nстатистика до обробки:")
        display(df_clean[comparison_cols].describe())
        print("\nстатистика після обробки:")
        display(df_no_outliers[comparison_cols].describe())
        
        print("\nприклад змінених значень:")
        for col in comparison_cols:
            changed_mask = df_clean[col] != df_no_outliers[col]
            if changed_mask.any():
                changed_df = pd.DataFrame({
                    f'{col}_до': df_clean.loc[changed_mask, col].head(),
                    f'{col}_після': df_no_outliers.loc[changed_mask, col].head()
                })
                display(changed_df)
else:
    print("\nаномалії не виявлені в безперервних числових ознаках")

print(f"\nрозмір після обробки: {df_no_outliers.shape}")

print("\n" + "="*80)
print("1d. масштабування ознак")
print("="*80)

scale_cols = df_no_outliers.select_dtypes(include=[np.number]).columns.tolist()
scale_cols = [col for col in scale_cols if col.lower() not in exclude_cols]

print(f"\nстовпці для масштабування ({len(scale_cols)}):")

df_scaled = df_no_outliers.copy()

print("\nстатистика до масштабування (перші 5 ознак):")
display(df_scaled[scale_cols[:5]].describe())

scaler = StandardScaler()
df_scaled[scale_cols] = scaler.fit_transform(df_scaled[scale_cols])

print("\nстатистика після масштабування (перші 5 ознак):")
display(df_scaled[scale_cols[:5]].describe())

print("\nметод: standardscaler (z-score нормалізація)")
print("формула: z = (x - μ) / σ")
print("результат: середнє ≈ 0, стандартне відхилення ≈ 1")

print("\nпорівняння перших 10 рядків (3 ознаки):")
comparison_before = df_no_outliers[scale_cols[:3]].head(10).copy()
comparison_before.columns = [f'{col}_до' for col in comparison_before.columns]
comparison_after = df_scaled[scale_cols[:3]].head(10).copy()
comparison_after.columns = [f'{col}_після' for col in scale_cols[:3]]
display(pd.concat([comparison_before, comparison_after], axis=1))

output_file = 'asthma_data_preprocessed.csv'
df_scaled.to_csv(output_file, index=False)

print("\n" + "="*80)
print(f"дані збережено: {output_file}")
print(f"фінальний розмір: {df_scaled.shape}")
print("\nфінальний датасет (перші 10 рядків):")
display(df_scaled.head(10))
print("="*80)

початкова інформація про датасет:
розмір: (2392, 30)
кількість записів: 2392
кількість ознак: 30

типи даних:
patientid                          int64
age                                int64
gender                              bool
ethnicity                          int64
educationlevel                     int64
bmi                              float64
smoking                             bool
physicalactivity                 float64
dietquality                      float64
sleepquality                     float64
pollutionexposure                float64
pollenexposure                   float64
dustexposure                     float64
petallergy                          bool
familyhistoryasthma                 bool
historyofallergies                  bool
eczema                              bool
hayfever                            bool
gastroesophagealreflux              bool
lungfunctionfev1                 float64
lungfunctionfvc                  float64
wheezing                     

,patientid,age,gender,ethnicity,educationlevel,bmi,smoking,physicalactivity,dietquality,sleepquality,...,lungfunctionfvc,wheezing,shortnessofbreath,chesttightness,coughing,nighttimesymptoms,exerciseinduced,diagnosis,doctorincharge,timestamp
0,5034,63,False,1,0,15.848744,False,0.894448,5.488696,8.701003,...,4.941206,False,False,True,False,False,True,False,Dr_Confid,2025-04-10 23:35:08
1,5035,26,True,2,2,22.757042,False,5.897329,6.341014,5.153966,...,1.702393,True,False,False,True,True,True,False,Dr_Confid,2025-08-30 18:23:59
2,5036,57,False,2,1,18.395396,False,6.739367,9.196237,6.840647,...,5.022553,True,True,True,False,True,True,False,Dr_Confid,2025-01-07 21:12:22
3,5037,40,True,2,1,38.515278,False,1.404503,5.826532,4.253036,...,2.300159,True,False,True,True,True,False,False,Dr_Confid,2025-11-01 18:22:40
4,5038,61,False,0,3,19.283802,False,4.604493,3.127048,9.625799,...,3.067944,True,True,True,False,False,True,False,Dr_Confid,2025-04-12 03:14:15



1a. аналіз та заміна пропущених значень

інформація про пропущені значення:


,стовпець,пропущені,відсоток



висновок: пропущені значення відсутні
не потрібно застосовувати методи імпутації

1b. кодування категоріальних ознак

категоріальні стовпці (16):

gender:
  кількість унікальних значень: 2
  значення: [False  True]
  → буде застосовано label encoding

smoking:
  кількість унікальних значень: 2
  значення: [False  True]
  → буде застосовано label encoding

petallergy:
  кількість унікальних значень: 2
  значення: [ True False]
  → буде застосовано label encoding

familyhistoryasthma:
  кількість унікальних значень: 2
  значення: [ True False]
  → буде застосовано label encoding

historyofallergies:
  кількість унікальних значень: 2
  значення: [False  True]
  → буде застосовано label encoding

eczema:
  кількість унікальних значень: 2
  значення: [False  True]
  → буде застосовано label encoding

hayfever:
  кількість унікальних значень: 2
  значення: [False  True]
  → буде застосовано label encoding

gastroesophagealreflux:
  кількість унікальних значень: 2
  значення: [False  True]
 

...

,age,gender,ethnicity,educationlevel,bmi
count,2.392000e+03,2.392000e+03,2.392000e+03,2.392000e+03,2.392000e+03
mean,1.396133e-16,-8.317390e-17,-7.129191e-17,5.940993e-18,2.354118e-16
std,1.000209e+00,1.000209e+00,1.000209e+00,1.000209e+00,1.000209e+00
min,-1.719180e+00,-9.867104e-01,-6.793009e-01,-1.455673e+00,-1.696231e+00
25%,-8.859290e-01,-9.867104e-01,-6.793009e-01,-3.421554e-01,-8.717301e-01
50%,-6.386399e-03,-9.867104e-01,-6.793009e-01,-3.421554e-01,-2.675999e-02
75%,8.731562e-01,1.013469e+00,3.349861e-01,7.713625e-01,8.765165e-01
max,1.706407e+00,1.013469e+00,2.363560e+00,1.884880e+00,1.769516e+00



метод: standardscaler (z-score нормалізація)
формула: z = (x - μ) / σ
результат: середнє ≈ 0, стандартне відхилення ≈ 1

порівняння перших 10 рядків (3 ознаки):


,age_до,gender_до,ethnicity_до,age_після,gender_після,ethnicity_після
0,63,0,1,0.965740,-0.986710,0.334986
1,26,1,2,-0.747054,1.013469,1.349273
2,57,0,2,0.687989,-0.986710,1.349273
3,40,1,2,-0.098970,1.013469,1.349273
4,61,0,0,0.873156,-0.986710,-0.679301
5,21,0,2,-0.978512,-0.986710,1.349273
6,45,1,1,0.132489,1.013469,0.334986
7,26,0,0,-0.747054,-0.986710,-0.679301
8,49,1,1,0.317656,1.013469,0.334986
9,45,1,1,0.132489,1.013469,0.334986



дані збережено: asthma_data_preprocessed.csv
фінальний розмір: (2392, 30)

фінальний датасет (перші 10 рядків):


,patientid,age,gender,ethnicity,educationlevel,bmi,smoking,physicalactivity,dietquality,sleepquality,...,lungfunctionfvc,wheezing,shortnessofbreath,chesttightness,coughing,nighttimesymptoms,exerciseinduced,diagnosis,doctorincharge,timestamp
0,5034,0.965740,-0.986710,0.334986,-1.455673,-1.582769,-0.406355,-1.432099,0.160113,0.971063,...,0.920608,-1.214986,-1.000836,0.993333,-1.006711,-1.230954,0.808131,0,Dr_Confid,2025-04-10 23:35:08
1,5035,-0.747054,1.013469,1.349273,0.771363,-0.623300,-0.406355,0.291269,0.453069,-1.076746,...,-1.564256,0.823055,-1.000836,-1.006711,0.993333,0.812378,0.808131,0,Dr_Confid,2025-08-30 18:23:59
2,5036,0.687989,-0.986710,1.349273,-0.342155,-1.229074,-0.406355,0.581330,1.434458,-0.102976,...,0.983019,0.823055,0.999164,0.993333,-1.006711,0.812378,0.808131,0,Dr_Confid,2025-01-07 21:12:22
3,5037,-0.098970,1.013469,1.349273,-0.342155,1.565307,-0.406355,-1.256398,0.276233,-1.596880,...,-1.105641,0.823055,-1.000836,0.993333,0.993333,0.812378,-1.237424,0,Dr_Confid,2025-11-01 18:22:40
4,5038,0.873156,-0.986710,-0.679301,1.884880,-1.105686,-0.406355,-0.154081,-0.651625,1.504976,...,-0.516586,0.823055,0.999164,0.993333,-1.006711,-1.230954,0.808131,0,Dr_Confid,2025-04-12 03:14:15
5,5039,-0.978512,-0.986710,1.349273,-1.455673,-0.754418,-0.406355,-1.578296,-1.121805,1.460789,...,1.655070,0.823055,-1.000836,0.993333,-1.006711,-1.230954,0.808131,0,Dr_Confid,2025-05-31 14:51:40
6,5040,0.132489,1.013469,0.334986,-0.342155,0.416809,2.460904,1.488132,0.690060,-0.734873,...,-1.564932,0.823055,0.999164,0.993333,-1.006711,-1.230954,-1.237424,0,Dr_Confid,2025-11-30 07:21:34
7,5041,-0.747054,-0.986710,-0.679301,-0.342155,-0.166172,2.460904,1.134119,-1.167394,-0.339373,...,0.207908,0.823055,-1.000836,-1.006711,0.993333,0.812378,0.808131,0,Dr_Confid,2025-11-04 03:12:02
8,5042,0.317656,1.013469,0.334986,0.771363,0.754338,-0.406355,-0.813488,-0.379062,-0.678574,...,1.112201,0.823055,0.999164,0.993333,0.993333,-1.230954,-1.237424,0,Dr_Confid,2025-05-03 17:37:37
9,5043,0.132489,1.013469,0.334986,-0.342155,0.370191,-0.406355,-0.742711,-0.830133,0.124644,...,1.557256,0.823055,-1.000836,-1.006711,-1.006711,0.812378,0.808131,0,Dr_Confid,2025-07-11 10:28:35
